# Test AAO with Conductor — a guided walkthrough

Run the cells **from top to bottom** using the ▶ button or **Shift+Enter**. You do not need to copy workflow IDs or read JSON.

This notebook connects to your real **local Conductor OSS server** and uses the existing synthetic banking adapters. All accounts, customers, and CRM writes are mocks.

### Before you start
1. Keep Docker Desktop and the Conductor stack running.
2. Stop any `aao workers` terminal with **Ctrl+C**. This notebook runs the workers itself when you execute a processing cell.
3. In VS Code, select **.venv** using **Select Kernel → Python Environments**.
4. If the kernel is missing, install the notebook dependencies once from the project terminal:
   `uv pip install --python .venv/Scripts/python.exe -r requirements-notebook.txt`

**Run All** tests everything automatically. For a clearer explanation, run one cell at a time and inspect the tables. Processing may take up to three minutes per case. Rerun the setup cell to create a new set of cases; scenario cells reuse their case within the current session.

| Scenario | What happens | Account created? |
|---|---|---|
| Automatic approval | Valid application; no human needed | Yes |
| Human approval | Wait for review, then approve | Yes |
| Human rejection | Wait for review, then reject | No |
| Invalid application | KYC validation fails | No |

## 1. Connect and prepare

This cell checks server health and creates local adapters. It does not start background threads.

The ledger is the same persistent `.local/effects.sqlite3` used by the CLI. The notebook polls shared AAO task queues, so use it only on this local demo server; older synthetic pending tasks can also be picked up.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "workflows" / "aao-v1.json").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the conductor-demo project folder.")
for folder in (ROOT / "src", ROOT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from conductor_demo.conductor import Conductor
from demo_helpers import Walkthrough, message, table

SERVER_URL = "http://localhost:8000/api"
demo = Walkthrough(ROOT, SERVER_URL)
demo.register()

## 2. Automatic approval

The customer has current KYC and does not need human review.

**Expected:** all service steps complete, the outcome is **APPROVED**, and exactly **one mock account** is created.

In [2]:
demo.start("Automatic approval", "automatic")
demo.wait("Automatic approval")
demo.check("Automatic approval", "APPROVED")

Processing local tasks… this may take up to three minutes.


What to check,Result
Execution status,COMPLETED
Business decision,APPROVED
Account,mock-2567436268fba582
Accounts created for this case,1
Validation issues,None


Journey step,Status
Classify request,COMPLETED
Extract application,COMPLETED
Validate through gateway,COMPLETED
Human review,Skipped
Open mock account,COMPLETED
Update CRM,COMPLETED
Complete case,COMPLETED


Case ID,Workflow ID
notebook-4be289a6ebea44d9a2623fdba9fc1f95,a69c78fb-fb12-42bc-85b5-dc0db6a963fa


## 3. Human approval — first observe the pause

This application is valid but needs a person to review it.

**Expected now:** execution is **RUNNING**, human review is **IN_PROGRESS**, and no account exists yet. RUNNING at this point means the workflow is safely waiting, not stuck.

After this cell finishes, you can leave the notebook idle. Conductor holds the wait; no notebook worker thread waits for a person.

In [3]:
demo.start("Human approval", "review")
demo.wait("Human approval", for_review=True);

Processing local tasks… this may take up to three minutes.


What to check,Result
Execution status,RUNNING
Business decision,Waiting for review
Account,None
Accounts created for this case,0
Validation issues,None


Journey step,Status
Classify request,COMPLETED
Extract application,COMPLETED
Validate through gateway,COMPLETED
Human review,IN_PROGRESS
Open mock account,Not reached
Update CRM,Not reached
Complete case,Not reached


Case ID,Workflow ID
notebook-0e513fa9d8ad48a59803687a3c3457be,6eef574a-6c9d-4595-bb65-33d0f62b1004


### The wait survives worker/client reconstruction

Recreate the notebook's service workers and SDK client, then inspect the same case. The human-review pause is stored by Conductor, not in these Python objects.

In [4]:
demo.client = Conductor(SERVER_URL)
demo.workers = demo.new_workers()
demo.show("Human approval");

What to check,Result
Execution status,RUNNING
Business decision,Waiting for review
Account,None
Accounts created for this case,0
Validation issues,None


Journey step,Status
Classify request,COMPLETED
Extract application,COMPLETED
Validate through gateway,COMPLETED
Human review,IN_PROGRESS
Open mock account,Not reached
Update CRM,Not reached
Complete case,Not reached


Case ID,Workflow ID
notebook-0e513fa9d8ad48a59803687a3c3457be,6eef574a-6c9d-4595-bb65-33d0f62b1004


### Approve the application

Run this cell when you are ready to simulate the reviewer's approval.

**Expected:** fulfillment, CRM update, and case completion finish. Exactly one mock account is created.

In [5]:
demo.decide("Human approval", "APPROVED")
demo.check("Human approval", "APPROVED")

Processing local tasks… this may take up to three minutes.


What to check,Result
Execution status,COMPLETED
Business decision,APPROVED
Account,mock-58eda48080835df1
Accounts created for this case,1
Validation issues,None


Journey step,Status
Classify request,COMPLETED
Extract application,COMPLETED
Validate through gateway,COMPLETED
Human review,COMPLETED
Open mock account,COMPLETED
Update CRM,COMPLETED
Complete case,COMPLETED


Case ID,Workflow ID
notebook-0e513fa9d8ad48a59803687a3c3457be,6eef574a-6c9d-4595-bb65-33d0f62b1004


## 4. Human rejection — observe a separate case

This uses the same valid application fixture but a **new case**. It also pauses for review.

In [6]:
demo.start("Human rejection", "review")
demo.wait("Human rejection", for_review=True);

Processing local tasks… this may take up to three minutes.


What to check,Result
Execution status,RUNNING
Business decision,Waiting for review
Account,None
Accounts created for this case,0
Validation issues,None


Journey step,Status
Classify request,COMPLETED
Extract application,COMPLETED
Validate through gateway,COMPLETED
Human review,IN_PROGRESS
Open mock account,Not reached
Update CRM,Not reached
Complete case,Not reached


Case ID,Workflow ID
notebook-4b1112c39c5c48b688d2244c5b38b4b9,f3ad0641-b26b-4380-936c-fd4b311fe032


### Reject this application

**Expected:** fulfillment is **skipped**, no account is created, and CRM records rejection before case completion.

The execution status is **COMPLETED** because the workflow handled the rejection successfully. The business outcome is **REJECTED**.

In [7]:
demo.decide("Human rejection", "REJECTED")
demo.check("Human rejection", "REJECTED")

Processing local tasks… this may take up to three minutes.


What to check,Result
Execution status,COMPLETED
Business decision,REJECTED
Account,None
Accounts created for this case,0
Validation issues,None


Journey step,Status
Classify request,COMPLETED
Extract application,COMPLETED
Validate through gateway,COMPLETED
Human review,COMPLETED
Open mock account,Skipped
Update CRM,COMPLETED
Complete case,COMPLETED


Case ID,Workflow ID
notebook-4b1112c39c5c48b688d2244c5b38b4b9,f3ad0641-b26b-4380-936c-fd4b311fe032


## 5. Invalid application

This customer's synthetic KYC is not current.

**Expected:** validation reports **KYC not current**. Human review and fulfillment are skipped; no account is created. CRM and case completion still finish with outcome **REJECTED**.

In [8]:
demo.start("Invalid application", "invalid")
demo.wait("Invalid application")
demo.check("Invalid application", "REJECTED")

Processing local tasks… this may take up to three minutes.


What to check,Result
Execution status,COMPLETED
Business decision,REJECTED
Account,None
Accounts created for this case,0
Validation issues,KYC not current


Journey step,Status
Classify request,COMPLETED
Extract application,COMPLETED
Validate through gateway,COMPLETED
Human review,Skipped
Open mock account,Skipped
Update CRM,COMPLETED
Complete case,COMPLETED


Case ID,Workflow ID
notebook-b80e701033484de389a013b63501f95f,5432ec79-3dc8-4579-9754-a9752e406af7


## 6. All results in one table

Each PASS checks the final decision, account reference, number of fulfillment records, CRM update, and case completion. No account created means zero fulfillment records for that case.

In [9]:
demo.summary()

Scenario,Expected,Actual,Accounts created,Test
Automatic approval,APPROVED,APPROVED,1,PASS
Human approval,APPROVED,APPROVED,1,PASS
Human rejection,REJECTED,REJECTED,0,PASS
Invalid application,REJECTED,REJECTED,0,PASS


## 7. What if approval is sent twice?

This repeats approval for the already approved human-review case.

**Expected:** the repeated decision is recognized without sending another update, and there is still one account. Attempting to change that approval to rejection is refused.

This demonstrates sequential decision protection. Concurrent reviewers still require the production review API described in the README.

In [10]:
demo.demonstrate_duplicate_review()

## 8. Focused checks for retries and mappings

These checks do not create cases. They also test concurrent mock CRM delivery and fulfillment result replay after adapter reconstruction, which the four scenario cards alone do not prove.

In [11]:
import subprocess
import os

checks = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "tests"), "-q"],
    cwd=ROOT, capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
)
details = (checks.stdout + checks.stderr).strip()
message("Focused tests passed" if checks.returncode == 0 else "Focused tests failed",
        details, "#15803d" if checks.returncode == 0 else "#dc2626")
assert checks.returncode == 0, details

## Reading the results

- **APPROVED:** the application passed, and a synthetic account was created.
- **REJECTED:** no account was created; inspect the validation issues or the review decision.
- **RUNNING + human review IN_PROGRESS:** waiting for a person. Run the approve/reject cell next.
- **COMPLETED:** Conductor finished the journey; inspect the business decision to distinguish approval from rejection.
- **FAILED / TIMED_OUT:** an execution problem, not a normal business rejection.

### If a cell fails
- **Missing module or kernel:** select the project's `.venv` kernel and install `requirements-notebook.txt`.
- **Conductor is not ready:** check Docker Compose status. Wait for `conductor-server` to be healthy, then rerun setup. Startup can take over a minute.
- **Still running after three minutes:** rerun that processing cell to continue the same case. Do not rerun setup unless you want new cases.
- **Cells run out of order:** rerun from setup using Run All.

No background workers remain after a processing cell finishes. Closing the notebook leaves unfinished review cases in Conductor. Synthetic cases and ledger results remain available; no cleanup or deletion runs automatically.

For architecture, service integration gaps, and local server recovery, see [README](../README.md).